In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, BooleanType,DataType

STORAGE_ACCOUNT = "adlsairbnbde"

In [0]:
spark.conf.set(
    "fs.azure.account.key.adlsairbnbde.dfs.core.windows.net",
    "KEY HERE"
)

BRONZE_VALIDATED_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/_validated/listings"
SILVER_PATH = f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/listings"

In [0]:
dbutils.fs.ls("abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/")

[FileInfo(path='abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/calendar/', name='calendar/', size=0, modificationTime=1785142714000),
 FileInfo(path='abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/listings/', name='listings/', size=0, modificationTime=1785142683000),
 FileInfo(path='abfss://bronze@adlsairbnbde.dfs.core.windows.net/airbnb/neighbourhoods/', name='neighbourhoods/', size=0, modificationTime=1785142820000)]

In [0]:
df = spark.read.format("delta").load(BRONZE_VALIDATED_PATH)

print(f"Rows read from Bronze : {df.count()}")
display(df.groupBy("_source_city", "_source_quarter").count())

Rows read from Bronze : 85028


_source_city,_source_quarter,count
lisbon,2025-Q3,25449
lisbon,2026-Q2,24876
barcelona,2025-Q3,19410
barcelona,2026-Q2,15293


In [0]:
df =df.withColumn(
    "price_cleaned",
    F.regexp_replace(F.col("price"),"[$,]","").cast(DoubleType()) )

price_null_count = df.filter(F.col("price_cleaned").isNull()).count()

total_count = df.count()
print(f"Rows with unparseable/missing price: {price_null_count} / {total_count} "
      f"({100 * price_null_count / total_count:.1f}%)")

Rows with unparseable/missing price: 11891 / 85028 (14.0%)


In [0]:
df = (
    df
    .withColumn("host_since_date", F.to_date("host_since"))
    .withColumn(
        "host_is_superhost_bool",
        F.when(F.col("host_is_superhost") == "t", True)
         .when(F.col("host_is_superhost") == "f", False)
         .otherwise(None)
    )
    .withColumn("host_listings_count_int", F.col("host_listings_count").cast(IntegerType()))
    .withColumn("accommodates_int", F.col("accommodates").cast(IntegerType()))
    .withColumn("bedrooms_int", F.col("bedrooms").cast(IntegerType()))
    .withColumn("number_of_reviews_int", F.col("number_of_reviews").cast(IntegerType()))
    .withColumn("review_scores_rating_double", F.col("review_scores_rating").cast(DoubleType()))
)

In [0]:
df = df.withColumn(
    "room_type_normalized",
    F.when(F.col("room_type").rlike("(?i)entire"), "Entire place")
     .when(F.col("room_type").rlike("(?i)private"), "Private room")
     .when(F.col("room_type").rlike("(?i)shared"), "Shared room")
     .when(F.col("room_type").rlike("(?i)hotel"), "Hotel room")
     .otherwise(F.col("room_type"))  
)

display(df.groupBy("room_type", "room_type_normalized").count().orderBy(F.desc("count")))

room_type,room_type_normalized,count
Entire home/apt,Entire place,59901
Private room,Private room,24078
Shared room,Shared room,525
Hotel room,Hotel room,524


In [0]:
df = df.withColumn(
    "has_reviews",
    F.when(F.col("number_of_reviews_int") > 0, True).otherwise(False)
)

In [0]:
before_dedupe = df.count()
df = df.dropDuplicates(["id", "_source_city", "_source_quarter"])
after_dedupe = df.count()
print(f"Rows before dedupe: {before_dedupe}, after: {after_dedupe}, removed: {before_dedupe - after_dedupe}")
 

Rows before dedupe: 85028, after: 85028, removed: 0


In [0]:
silver_listings = df.select(
    F.col("id").alias("listing_id"),
    F.col("host_id"),
    F.col("host_since_date"),
    F.col("host_is_superhost_bool").alias("host_is_superhost"),
    F.col("host_listings_count_int").alias("host_listings_count"),
    F.col("neighbourhood_cleansed").alias("neighbourhood"),
    F.col("room_type_normalized").alias("room_type"),
    F.col("accommodates_int").alias("accommodates"),
    F.col("bedrooms_int").alias("bedrooms"),
    F.col("bathrooms_text"),
    F.col("price_cleaned").alias("price"),
    F.col("number_of_reviews_int").alias("number_of_reviews"),
    F.col("has_reviews"),
    F.col("review_scores_rating_double").alias("review_scores_rating"),
    F.col("_source_city").alias("city"),
    F.col("_source_quarter").alias("quarter_label"),
    F.col("_ingested_at"),
)
 
display(silver_listings.limit(10))

listing_id,host_id,host_since_date,host_is_superhost,host_listings_count,neighbourhood,room_type,accommodates,bedrooms,bathrooms_text,price,number_of_reviews,has_reviews,review_scores_rating,city,quarter_label,_ingested_at
10003263,34111537,2015-05-24,false,1,el Poble Sec,Private room,2,null,1 bath,null,8,true,4.5,barcelona,2025-Q3,2026-07-27T11:06:55.907419
1000447810456915898,5107063,2013-02-16,false,12,Can Baró,Entire place,4,1,1 bath,125.0,47,true,4.72,barcelona,2025-Q3,2026-07-27T11:06:55.907419
1000514588274707195,448276178,2022-03-06,false,1,el Raval,Private room,2,1,Shared half-bath,51.0,72,true,4.31,barcelona,2025-Q3,2026-07-27T11:06:55.907419
1000518750453447385,541500884,null,false,1,Olivais,Entire place,6,3,1 bath,232.33,16,true,4.81,lisbon,2026-Q2,2026-07-27T11:06:57.495959
1000720,5501549,2013-03-17,false,2,el Guinardó,Private room,1,1,1 shared bath,66.0,5,true,4.2,barcelona,2025-Q3,2026-07-27T11:06:55.907419
1000837114643475255,541210371,null,true,11,Campo de Ourique,Entire place,2,null,1 bath,127.0,51,true,4.86,lisbon,2026-Q2,2026-07-27T11:06:57.495959
1001158603293748981,2246766,null,true,4,el Besòs i el Maresme,Entire place,6,3,1 bath,null,1,true,5.0,barcelona,2026-Q2,2026-07-27T11:06:56.507902
1001203296586815064,532435579,null,false,15,Pvoa de Santo Adrio e Olival de Basto,Entire place,4,1,1 bath,103.0,14,true,4.64,lisbon,2026-Q2,2026-07-27T11:06:57.495959
1001211276807625622,532435579,null,false,15,Pvoa de Santo Adrio e Olival de Basto,Entire place,4,1,1 bath,114.5,2,true,5.0,lisbon,2026-Q2,2026-07-27T11:06:57.495959
1001214024965449564,532435579,2023-08-17,false,15,Pvoa de Santo Adrio e Olival de Basto,Private room,2,1,1 private bath,80.0,0,false,null,lisbon,2025-Q3,2026-07-27T11:06:57.018903


In [0]:
(
    silver_listings.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("city")
    .option("overwriteSchema", "true")
    .save(SILVER_PATH)
)
 
print(f"Silver listings written to: {SILVER_PATH}")
print(f"Total rows: {silver_listings.count()}")
display(silver_listings.groupBy("city", "quarter_label").count())

Silver listings written to: abfss://silver@adlsairbnbde.dfs.core.windows.net/listings
Total rows: 85028


city,quarter_label,count
barcelona,2025-Q3,19410
barcelona,2026-Q2,15293
lisbon,2025-Q3,25449
lisbon,2026-Q2,24876


In [0]:
spark.read.format("delta").load("abfss://silver@adlsairbnbde.dfs.core.windows.net/listings") \
    .groupBy("city", "quarter_label").count().show()

+---------+-------------+-----+
|     city|quarter_label|count|
+---------+-------------+-----+
|   lisbon|      2025-Q3|25449|
|   lisbon|      2026-Q2|24876|
|barcelona|      2025-Q3|19410|
|barcelona|      2026-Q2|15293|
+---------+-------------+-----+

